# SCAMROAD — DEDICATED COLAB PRO TRAINING RECOVERY HUB

This notebook is specifically designed for Google Colab Pro sessions to:
1. **Audit & Discover**: Automatically search Google Drive (`/content/drive/MyDrive/ScamRoad`) for previous checkpoints (`last.pt`, `best.pt`, `epoch*.pt`).
2. **Validate Integrity**: Verify checkpoint serialization, optimizer state, model weights, and completed epochs.
3. **Restore Dataset Cache**: Instantaneously restore dataset splits from Google Drive NVMe cache without re-downloading 10 GB.
4. **Auto-Resume**: Seamlessly continue interrupted training from the exact completed epoch without restarting from epoch 1.


## Step 1: Hardware & Cloud GPU Verification
Inspect the assigned GPU, VRAM, system RAM, and ephemeral storage.

In [ ]:
# 1. Hardware Diagnostics
!nvidia-smi

import psutil, shutil, torch
ram = psutil.virtual_memory()
disk = shutil.disk_usage('/content')
print('=' * 65)
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU       : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM       : {vram:.2f} GB')
print(f'System RAM       : {ram.total / (1024**3):.2f} GB (Free: {ram.available / (1024**3):.2f} GB)')
print(f'Disk Space       : {disk.total / (1024**3):.2f} GB (Free: {disk.free / (1024**3):.2f} GB)')
print('=' * 65)


## Step 2: Mount Persistent Google Drive Storage
Mount Google Drive to access persistent runs, checkpoints, and cached dataset splits.

In [ ]:
# 2. Google Drive Mount
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive/ScamRoad')
drive_root.mkdir(parents=True, exist_ok=True)

for folder in ['datasets', 'weights', 'checkpoints', 'exports', 'runs', 'logs', 'dataset_artifacts']:
    (drive_root / folder).mkdir(exist_ok=True)

print(f'[SUCCESS] Persistent Google Drive storage verified at: {drive_root}')


## Step 3: Clone or Synchronize ScamRoad Repository
Pulls the latest verified codebase containing the auto-resume engine.

In [ ]:
# 3. Clone or pull repo
import os, sys
from pathlib import Path

repo_path = Path('/content/potholes')
if not repo_path.exists():
    !git clone https://github.com/Priya-Ranjan-0201/potholes.git /content/potholes
else:
    %cd /content/potholes
    !git pull origin main

%cd /content/potholes
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

print(f'[SUCCESS] Codebase active at: {Path.cwd()}')


## Step 4: Install Dependencies

In [ ]:
# 4. Install training dependencies
!pip install -q ultralytics onnx onnxruntime-gpu pyyaml psutil albumentations


## Step 5: Checkpoint Discovery & Diagnostic Audit
Recursively scans Google Drive and Colab storage for `*.pt`, `*.pth`, and `*.ckpt` files, validating each file's integrity.

In [ ]:
# 5. Checkpoint Discovery & Verification
from pathlib import Path
from scripts.colab.setup_colab import verify_checkpoint, PipelineState

drive_root = Path('/content/drive/MyDrive/ScamRoad')
runs_dir = drive_root / 'runs'
state_file = drive_root / 'pipeline_state.json'

# Load persistent state
state = PipelineState(state_file)
state.print_summary()

# Search for checkpoints
print('=' * 75)
print('                 SCAMROAD CHECKPOINT DISCOVERY REPORT                 ')
print('=' * 75)
found_ckpts = list(runs_dir.rglob('*.pt')) + list(Path('/content').glob('*.pt'))
if not found_ckpts:
    print('[INFO] No prior checkpoint files found in Google Drive or /content.')
    print('       A fresh, resumable training session will be initialized.')
else:
    print(f'Found {len(found_ckpts)} checkpoint file(s):\n')
    for ckpt in found_ckpts:
        meta = verify_checkpoint(ckpt)
        valid_str = '✔ VALID' if meta['valid'] else '✖ CORRUPT'
        epoch_str = f'Epoch {meta["epoch"] + 1}' if meta["epoch"] >= 0 else 'Final/Exported'
        opt_str = 'With Optimizer' if meta['has_optimizer'] else 'No Optimizer'
        print(f' • {ckpt.relative_to(drive_root) if str(ckpt).startswith(str(drive_root)) else ckpt}')
        print(f'   Status: {valid_str} | Completed: {epoch_str} | Size: {meta["file_size_mb"]} MB | {opt_str}')
        if meta.get('error'):
            print(f'   Error: {meta["error"]}')
print('=' * 75 + '\n')


## Step 6: Instant Dataset Cache Restoration
Restores pre-split dataset from Google Drive (`/content/drive/MyDrive/ScamRoad/datasets/splits`) to local NVMe (`/content/data/splits`) without re-downloading.

In [ ]:
# 6. Dataset Cache Restore
from pathlib import Path
from scripts.colab.setup_colab import restore_dataset_from_drive

drive_splits = Path('/content/drive/MyDrive/ScamRoad/datasets/splits')
local_splits = Path('/content/data/splits')

if (drive_splits / 'dataset.yaml').exists():
    print('[INFO] Found cached dataset splits on Google Drive. Restoring to local NVMe...')
    restore_dataset_from_drive(drive_splits, local_splits)
    print(f'[SUCCESS] Dataset splits active at: {local_splits / "dataset.yaml"}')
elif (local_splits / 'dataset.yaml').exists():
    print(f'[INFO] Dataset splits already present locally at {local_splits}.')
else:
    print('[WARNING] No cached dataset found on Google Drive.')
    print('          Please run Step 6 in the main pipeline notebook to ingest datasets.')


## Step 7: Resume Interrupted Training Session
Resumes training from the exact completed epoch. If training is already completed, it cleanly skips.

In [ ]:
# 7. Resume Training Run
from pathlib import Path
from scripts.colab.setup_colab import safe_train_with_auto_resume, state

drive_root = Path('/content/drive/MyDrive/ScamRoad')
runs_dir = drive_root / 'runs'
dataset_cfg = Path('/content/data/splits/dataset.yaml')

# Select model to resume
MODEL_KEY = 'model_yolo11n'  # Options: 'model_yolo11n', 'model_yolo11s', 'model_yolo11n_seg', 'model_rtdetr_l'
BASE_WEIGHT = 'yolo11n.pt'

results = safe_train_with_auto_resume(
    model_key=MODEL_KEY,
    base_weight=BASE_WEIGHT,
    dataset_cfg=dataset_cfg,
    runs_dir=runs_dir,
    epochs=50,
    batch=16,
    imgsz=640,
    device=0,
    state=state,
    drive_splits_dir=drive_root / 'datasets' / 'splits',
    local_splits_dir=Path('/content/data/splits'),
    save_period=1,
)


## Step 8: Validate & Export Best Weights to ONNX
Exports the recovered best weights to ONNX format with half-precision support for edge deployment.

In [ ]:
# 8. Export to ONNX
from ultralytics import YOLO
from pathlib import Path

best_pt = drive_root / 'runs' / MODEL_KEY / 'weights' / 'best.pt'
if best_pt.exists():
    print(f'[INFO] Exporting best model: {best_pt}...')
    model = YOLO(str(best_pt))
    onnx_path = model.export(format='onnx', dynamic=True, simplify=True)
    print(f'[SUCCESS] Exported model to ONNX: {onnx_path}')
else:
    print(f'[INFO] Best weights {best_pt} not yet generated (training in progress).')
